# Time-Series Forecasting — Walk-Forward CV, Lag Features, and Baseline Models

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" alt="MGMT 474 AI Logo" width="120"/>
</div>
</center>

<center>

# <center>MGMT47400 Predictive Analytics</center>

# <center>Notebook 16</center>

</center>

<center>

<a href="https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb16_time_series_forecasting_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

</center>

<hr>


## Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish a forecasting problem from a generic supervised-learning problem and choose the right evaluation protocol.
2. Build a time-respecting train/test split where the test window is the most recent slice of history.
3. Run **walk-forward cross-validation** with `TimeSeriesSplit` instead of k-fold (which would shuffle time and leak the future into the past).
4. Engineer **lag features** (last month, 12 months ago for seasonality) and fit a linear regression that respects temporal order.
5. Compare three baselines — naive, seasonal-naive, and lag-feature linear regression — using a single error metric (MAE) on identical CV folds.
6. Open the locked test window in a one-shot evaluation ceremony, mirroring nb14's protocol but adapted to time.


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. Complete both to receive participation credit.


## 💼 Why This Matters: Forecasting Monthly Demand for the Operations Team

The **VP of Operations at DemandCo** is preparing the next fiscal year's procurement plan. Her team places quarterly orders to the supplier; over-order and inventory cost balloons, under-order and the company stocks out for weeks. She has 60 months of historical demand and one question:

> *"Can we forecast next month's demand with enough confidence to set a procurement target — and how do we measure 'enough confidence' before we trust it?"*

This is **not** the kind of problem we solved in nb01–nb15. There, every row was an independent observation and a 60/20/20 random split was the right protocol. Here, the rows are months in a sequence — the **order matters**, and shuffling them would let the model peek at the future during training (a classic data leak). The fix is structural: the test window is always the **most recent** slice of history, and cross-validation walks forward in time.

This notebook teaches the four ideas the operations team needs:

1. **Time-respecting splits** — never shuffle.
2. **Walk-forward CV** (`TimeSeriesSplit`) — every fold trains on the past, validates on the future.
3. **Lag features** — last month and 12 months ago are the cheapest, most useful predictors for any business series.
4. **Honest baselines** — naive (last value) and seasonal-naive (12 months ago) are surprisingly hard to beat. If a complex model does not beat them, it is not worth shipping.

**A question that often comes up here:** *"Why isn't this just nb14 with a different metric?"* Two reasons. First, k-fold CV with shuffled rows would let row 50 be in the training fold and row 49 in the validation fold — the model would see a future month while learning to predict an earlier one. That defeats the entire idea of forecasting. Second, demand series have **seasonality** (holidays, fiscal calendars), so a feature engineered as "value 12 months ago" is structurally meaningful in a way that "row 12 in the dataset" is not. The mechanics of CV-first evaluation carry over; the data structure forces new tooling.


## 1. Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.precision', 3)
print("Setup complete!")
print(f"RANDOM_SEED = {RANDOM_SEED}")


**Reading the output:** A clean `Setup complete!` confirms the libraries we need are available. The two new tools today are `TimeSeriesSplit` (walk-forward CV) and lag features (we will build them by hand with `pd.DataFrame.shift`).


## 2. The DemandCo Series — Generate and Visualize

We simulate **60 months** of demand: a steady linear trend (sales grow ~2 units/month), an annual seasonality (peak in December, trough in February — a typical retail pattern), and Gaussian noise (everything random the model cannot explain). The values land in the 80–250 unit range, the rough scale of DemandCo's monthly orders.


In [ ]:
# 60 months of synthetic demand
n_months = 60
months = pd.date_range("2021-01-01", periods=n_months, freq="MS")
trend = np.arange(n_months) * 2.0  # +2 units / month
seasonality = 30 * np.sin(2 * np.pi * (np.arange(n_months) % 12) / 12 - np.pi/2)
noise = np.random.normal(0, 8, n_months)
demand = 120 + trend + seasonality + noise

df = pd.DataFrame({"month": months, "demand": demand})
df = df.set_index("month")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df.index, df["demand"], marker="o", color="#1f77b4")
ax.set_xlabel("Month")
ax.set_ylabel("Demand (units)")
ax.set_title("DemandCo Monthly Demand — 60 months")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(df.head())


**Reading the output:**

The time plot reveals two structural features the operations team can use: a **steady upward trend** (the level rises across the five years) and a **regular seasonal swing** (peak ~12 months apart). The noise is real but small relative to the signal — a forecast that captures trend + seasonality should land within a few units of truth on average.

**A question that often comes up here:** *"How would I know there's seasonality without simulating it?"* For a real series, plot it first (a time plot). If you see repeating bumps at a regular calendar interval — months, weeks, hours — you have seasonality. Formal tools include the **autocorrelation function** (`statsmodels.tsa.stattools.acf`), which spikes at the seasonal lag, and the **STL decomposition** which splits the series into trend, seasonal, and residual components. We skip the formal tools today because the visual signal is already strong; the FPP textbook (linked in the bibliography) is the deep dive.


## 3. Time-Respecting Train/Test Split

Hold out the **last 12 months** (the most recent year) as the locked test window. Fit and select on months 1–48. This is the time-series analog of nb14's locked test set: do not touch it until the final evaluation ceremony in section 8.


In [ ]:
TEST_HORIZON = 12
df_train = df.iloc[:-TEST_HORIZON].copy()
df_test = df.iloc[-TEST_HORIZON:].copy()
print(f"Training: {df_train.index.min().date()} -> {df_train.index.max().date()} ({len(df_train)} months)")
print(f"Test    : {df_test.index.min().date()} -> {df_test.index.max().date()} ({len(df_test)} months) [LOCKED]")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df_train.index, df_train["demand"], color="#1f77b4", label="Train (selection)")
ax.plot(df_test.index, df_test["demand"], color="#d62728", linestyle="--", label="Test (locked)")
ax.axvline(df_train.index.max(), color="grey", linestyle=":", alpha=0.7)
ax.set_title("Time-Respecting Split: train ends, test begins")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the output:**

The dashed red segment is the locked test window — every model selection decision below uses only the solid blue training segment. This mirrors nb14's discipline: the test set is touched **once**, in section 8, after the champion is chosen.


## 4. Walk-Forward Cross-Validation with `TimeSeriesSplit`

`TimeSeriesSplit(n_splits=5)` produces 5 folds where every fold's training data comes **before** its validation data, and the training window grows over time. This is the structural fix that makes evaluation honest in a temporal setting.


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

fig, ax = plt.subplots(figsize=(11, 4))
for fold, (train_idx, val_idx) in enumerate(tscv.split(df_train)):
    ax.plot(train_idx, [fold] * len(train_idx), "s", color="#1f77b4", markersize=6, label="train" if fold == 0 else "")
    ax.plot(val_idx, [fold] * len(val_idx), "s", color="#ff7f0e", markersize=6, label="val" if fold == 0 else "")
ax.set_yticks(range(5))
ax.set_yticklabels([f"fold {i+1}" for i in range(5)])
ax.set_xlabel("Month index in training data")
ax.set_title("Walk-Forward CV: train (blue) always precedes val (orange)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


**Reading the output:**

Each row is one CV fold. Blue squares are training months; orange squares are validation months. Two structural facts to internalize:

1. **Orange always sits to the right of blue** — the model never sees a future month while learning to predict an earlier one.
2. **The training window grows** — fold 1 trains on the first 8 months and validates on the next 8; fold 5 trains on the first 40 and validates on the last 8. This mimics what really happens in deployment: you accumulate more history over time.

**A question that often comes up here:** *"Why not pick a fixed-size window for training (say, always 24 months)?"* You can — that variant is called the **rolling-window** strategy and `TimeSeriesSplit(max_train_size=...)` supports it. Use it when you suspect the relationship is non-stationary (e.g., a structural break after a market shift) and old data is misleading. The default expanding window is the right place to start because it uses every data point you have.


## 5. Build Lag Features

The cheapest, most useful features for any business time series are **lags**: yesterday's value, last month's value, last year's value. We build two:

- `lag1` — demand one month ago (autoregressive structure)
- `lag12` — demand 12 months ago (annual seasonality)

A linear regression on `[lag1, lag12]` is a strong, interpretable baseline that captures both short-term dependence and seasonality without any deep-learning machinery.


In [ ]:
def add_lags(frame, lags=(1, 12)):
    out = frame.copy()
    for L in lags:
        out[f"lag{L}"] = out["demand"].shift(L)
    return out.dropna()

df_train_lag = add_lags(df_train)
print(df_train_lag.head())
print(f"\nRows usable after lags: {len(df_train_lag)} (lost first 12 to missing lag12)")


**Reading the output:**

The `dropna()` at the bottom of `add_lags` drops the first 12 months because they have no `lag12` value (we cannot compute "12 months ago" for them). This is the cost of every lag feature: you sacrifice the earliest rows. For 60 months and a 12-month lag, that is a manageable loss; for daily data with a 365-day lag, you would lose the first year. This trade-off is one of the planning-doc decisions every forecaster makes.


## 📝 PAUSE-AND-DO Exercise 1 — Walk-Forward CV with Lag Features (10 minutes)

**Task:** Cross-validate a `LinearRegression` on `[lag1, lag12]` using `TimeSeriesSplit(5)` and report the mean MAE across folds.

**Hints:**
- `df_train_lag` already has `lag1`, `lag12`, and `demand`.
- For each `(train_idx, val_idx)` from `tscv.split(df_train_lag)`, slice the rows, fit on training, predict on validation, score with `mean_absolute_error`.
- Append each fold MAE to a list and print `np.mean(...)` and `np.std(...)`.

Type your code in the cell below.


In [ ]:
# YOUR SOLUTION CODE HERE

# Hints:
# X = df_train_lag[["lag1", "lag12"]].values
# y = df_train_lag["demand"].values
# tscv = TimeSeriesSplit(n_splits=5)
# fold_maes = []
# for tr, va in tscv.split(X):
#     ...


## 6. Three Baselines, Identical CV Folds

We now compare three forecasters on the **same** walk-forward folds, so the comparison is honest:

1. **Naive** — predict next month = last month. (`lag1`)
2. **Seasonal-naive** — predict next month = same month last year. (`lag12`)
3. **Linear regression on `[lag1, lag12]`** — let the model weight short-term and seasonal information.


In [ ]:
def cv_score(predict_fn, df_lag, splits):
    maes = []
    for tr, va in splits.split(df_lag):
        train, val = df_lag.iloc[tr], df_lag.iloc[va]
        y_pred = predict_fn(train, val)
        maes.append(mean_absolute_error(val["demand"], y_pred))
    return np.array(maes)

def predict_naive(train, val):       return val["lag1"].values
def predict_seasonal(train, val):    return val["lag12"].values
def predict_linear(train, val):
    m = LinearRegression().fit(train[["lag1", "lag12"]], train["demand"])
    return m.predict(val[["lag1", "lag12"]])

splits = TimeSeriesSplit(n_splits=5)
results = pd.DataFrame({
    "Naive (lag1)":          cv_score(predict_naive, df_train_lag, splits),
    "Seasonal-naive (lag12)": cv_score(predict_seasonal, df_train_lag, splits),
    "Linear [lag1, lag12]":   cv_score(predict_linear, df_train_lag, splits),
})
summary = pd.DataFrame({
    "MAE_mean": results.mean(),
    "MAE_std":  results.std(ddof=1),
    "MAE_95%CI_halfwidth": results.std(ddof=1) / np.sqrt(len(results)) * 2.776,
}).sort_values("MAE_mean")
print(summary)


**Reading the output:**

The lowest `MAE_mean` is the candidate champion. The `MAE_95%CI_halfwidth` (computed with the Student's *t* critical value at 4 degrees of freedom, exactly as in nb08) tells you whether the gap to the second-place model is **statistically distinguishable** or just CV noise. If the linear model's mean MAE is below the seasonal-naive's mean **and** the confidence intervals do not overlap, the linear model is the clear winner. If they overlap, the operations team should prefer the simpler model — there is no cost to using `lag12` directly as a forecast.

**A question that often comes up here:** *"Should I tune the linear regression?"* For a 2-feature linear regression, no — there is nothing to tune. The next step on this problem would be **adding more features** (`lag2`, `lag3`, calendar dummies for month-of-year, exogenous variables like price), not tuning hyperparameters. Forecasting champions are usually built by enriching the feature set, not by switching to fancier models.


## 7. Visualize the Champion's Forecast

Refit the champion on the **full** training data and plot its in-sample fit alongside the actual values. (We are still in selection — no test data touched.)


In [ ]:
champion = LinearRegression().fit(
    df_train_lag[["lag1", "lag12"]], df_train_lag["demand"]
)
df_train_lag["pred"] = champion.predict(df_train_lag[["lag1", "lag12"]])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df_train_lag.index, df_train_lag["demand"], "o-", color="#1f77b4", label="Actual")
ax.plot(df_train_lag.index, df_train_lag["pred"], "s--", color="#2ca02c", label="Linear [lag1, lag12]")
ax.set_title("Champion fit on training data (12 months lost to lags)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Coefficients: lag1 = {champion.coef_[0]:.3f}, lag12 = {champion.coef_[1]:.3f}")
print(f"Intercept   : {champion.intercept_:.3f}")


**Reading the output:**

The green dashed line tracks the blue actuals closely — that is the visual confirmation that lag1 + lag12 carry most of the forecastable signal. The coefficients tell the operations team a clean story: each unit of last month's demand contributes its coefficient × itself to next month's prediction; each unit of demand 12 months ago contributes its coefficient × itself. Add the intercept and you have a forecast that any analyst can verify by hand.


## 8. Opening the Locked Test Window — One-Shot Evaluation

We now do the time-series analog of nb14's "test-set opening ceremony." Predict the 12 locked months using lag values that **come from inside the locked test data only when they are not in the future**. The first locked month's `lag1` is the last training month's actual; its `lag12` is 12 months earlier in training.


In [ ]:
# Build lag features for the test window using actuals from train+test
df_full = pd.concat([df_train, df_test])
df_full = add_lags(df_full)
df_test_lag = df_full.loc[df_test.index]

y_test_pred = champion.predict(df_test_lag[["lag1", "lag12"]])
test_mae = mean_absolute_error(df_test_lag["demand"], y_test_pred)

cv_mean = summary.loc["Linear [lag1, lag12]", "MAE_mean"]
cv_low  = cv_mean - summary.loc["Linear [lag1, lag12]", "MAE_95%CI_halfwidth"]
cv_high = cv_mean + summary.loc["Linear [lag1, lag12]", "MAE_95%CI_halfwidth"]

verdict = ("INSIDE the CV 95% CI" if cv_low <= test_mae <= cv_high
           else "ABOVE the CV 95% CI" if test_mae > cv_high
           else "BELOW the CV 95% CI")

print(f"CV MAE 95% CI: [{cv_low:.2f}, {cv_high:.2f}]")
print(f"Test MAE     : {test_mae:.2f}  ->  {verdict}")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df_train.index, df_train["demand"], color="#1f77b4", label="Train")
ax.plot(df_test_lag.index, df_test_lag["demand"], "o-", color="#d62728", label="Test (actual)")
ax.plot(df_test_lag.index, y_test_pred, "s--", color="#2ca02c", label="Forecast")
ax.set_title("Locked Test Window — One-Shot Evaluation")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the output:**

The verdict mirrors nb14's protocol: if test MAE lands **inside** the CV 95% CI, the CV-based selection generalized; **above** the CI signals overfitting on the training history; **below** is unusual but possible (the locked window happened to be easier than the average CV fold). Either way, this is the **one** time the test window is touched. Any further model iteration would require collecting new history.


## 📝 PAUSE-AND-DO Exercise 2 — Add a Feature, Rerun the Comparison (10 minutes)

**Task:** Engineer a `month_of_year` calendar feature (1–12) and refit the linear baseline as `LinearRegression on [lag1, lag12, month_of_year]`. Run the same walk-forward CV comparison. Does the new feature beat the 2-lag baseline?

**Hints:**
- `df_train_lag["month_of_year"] = df_train_lag.index.month`
- Update `predict_linear_v2` to fit on the three features.
- Add a fourth row to the comparison table.
- Compare the two linear-model MAE means and their CIs — overlapping CIs means the new feature did not help.

Type your code in the cell below.


In [ ]:
# YOUR SOLUTION CODE HERE

# Hints:
# df_train_lag["month_of_year"] = df_train_lag.index.month
# def predict_linear_v2(train, val):
#     m = LinearRegression().fit(train[["lag1", "lag12", "month_of_year"]], train["demand"])
#     return m.predict(val[["lag1", "lag12", "month_of_year"]])
# results["Linear v2 [+month_of_year]"] = cv_score(predict_linear_v2, df_train_lag, splits)


## 9. Wrap-Up — Key Takeaways

1. **Forecasting is supervised learning with one structural rule: never let the future leak into the past.** That single rule changes the train/test split (recent slice held out), the cross-validation strategy (`TimeSeriesSplit`), and what counts as a feature (lags, not random shuffling).
2. **Naive baselines are surprisingly hard to beat.** If your fancy model does not beat seasonal-naive on identical CV folds with non-overlapping CIs, you do not have a champion — you have noise.
3. **The cost of lag features is the loss of the earliest rows.** A 12-month seasonal lag costs you the first year of history. Plan for it.
4. **Walk-forward CV is the time-series spine** of CV-first evaluation, exactly like `StratifiedKFold` was the classification spine in nb08–nb14.

**A question that often comes up here:** *"Where do RNNs and transformers fit?"* They are alternatives to lag-feature linear models when (a) the series is long enough (thousands of points, not 60), (b) the dependence is highly non-linear, and (c) you can spare an order of magnitude more compute. For business problems with a few years of monthly history, a well-engineered lag-feature linear regression or gradient-boosted model is almost always the right starting point — and often the right ending point. Deep learning shows up properly in nb19.

**Next stop — nb17: Data Communication and Poster Design.** Now that you have a forecast and a defensible CV-based comparison, the question becomes how to **communicate** it: the six principles of data communication, the eleven-section poster architecture for the M4 deliverable, and the data-ink-ratio cleanup that turns a notebook plot into a poster figure.


## Participation Assignment Submission Instructions

1. **Complete both PAUSE-AND-DO exercises** (sections 5 and 8).
2. **Run all cells** (`Runtime → Run all`).
3. **Save with output** (`File → Download → Download .ipynb`).
4. **Submit to Brightspace** as `nb16_time_series_forecasting_<your_lastname>.ipynb`.

**Bibliography**
- Hyndman & Athanasopoulos: *Forecasting: Principles and Practice* (the [free online textbook](https://otexts.com/fpp3/) is the deep dive on every concept above).
- scikit-learn User Guide: `TimeSeriesSplit` and time-series cross-validation.
- Course slides: `lecture_slides/08_time_series/`.


<center>

# Thank you!

</center>
